# TFM — Anonimización y Privacidad en IA Sanitaria

Pipeline experimental completo asociado a la memoria. El cuaderno reproduce
las siete fases descritas en el Capítulo 3 sobre el conjunto Diabetes 130-US
hospitals (UCI Machine Learning Repository).

## Fases

1. Ingesta y limpieza del conjunto clínico.
2. Perfilado de privacidad y cálculo del riesgo basal.
3. Reducción de dimensionalidad mediante agrupación semántica de códigos CIE-9.
4. Establecimiento del *baseline* sobre cuatro modelos de referencia.
5. Aplicación de *k*-anonimidad mediante ARX y evaluación de la utilidad.
6. Aplicación de Privacidad Diferencial mediante `diffprivlib`.
7. Auditoría adversarial mediante ataques de inferencia de membresía con ART.


## 1. Configuración del entorno

In [ ]:
!pip install -q diffprivlib==0.6.6 adversarial-robustness-toolbox==1.20.1 ucimlrepo

In [ ]:
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, f1_score
from scipy.stats import wilcoxon, chi2

import diffprivlib.models as dpm
from art.attacks.inference.membership_inference import MembershipInferenceBlackBox
from art.estimators.classification.scikitlearn import ScikitlearnLogisticRegression
from ucimlrepo import fetch_ucirepo

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
TEST_SIZE = 0.20
K_VALUES = [2, 5, 10, 25, 50]
EPSILON_VALUES = [0.1, 0.5, 1.0, 5.0, 10.0]
N_REPETITIONS_DP = 5
WORKDIR = Path(".")

## 2. Ingesta y limpieza del conjunto Diabetes 130-US

El conjunto de datos contiene 101.766 registros de encuentros hospitalarios
de pacientes diagnosticados con diabetes. Se aplican tres estrategias de
preprocesamiento para gestionar los valores ausentes: supresión de la
variable `weight` (96,85 % de nulos), imputación semántica de las pruebas
clínicas no realizadas y eliminación de filas con nulos residuales en
variables troncales.

In [ ]:
dataset = fetch_ucirepo(id=296)
df = pd.concat([dataset.data.features, dataset.data.targets], axis=1)
df.replace("?", np.nan, inplace=True)

df.drop(columns=["weight"], inplace=True)
df[["max_glu_serum", "A1Cresult", "medical_specialty", "payer_code"]] = (
    df[["max_glu_serum", "A1Cresult", "medical_specialty", "payer_code"]].fillna("No_Registrado")
)
df.dropna(subset=["race", "diag_1", "diag_2", "diag_3"], inplace=True)
print(f"Dimensiones tras limpieza: {df.shape}")

## 3. Perfilado de privacidad: riesgo basal de reidentificación

Se identifican cinco cuasi-identificadores (QIDs) consistentes con un
modelo de atacante realista con acceso a registros administrativos. El
riesgo basal se cuantifica mediante el tamaño mínimo de las clases de
equivalencia: un valor $k=1$ indica vulnerabilidad extrema a ataques de
enlace.

In [ ]:
qids = ["race", "gender", "age", "time_in_hospital", "admission_type_id"]
equivalence_classes = df.groupby(qids).size().reset_index(name="size")

n_total = len(df)
n_unique = equivalence_classes[equivalence_classes["size"] == 1]["size"].sum()
n_vulnerable_k5 = equivalence_classes[equivalence_classes["size"] < 5]["size"].sum()

print(f"Combinaciones únicas de QIDs: {len(equivalence_classes)}")
print(f"Pacientes con k=1 (vulnerabilidad extrema): {n_unique} ({n_unique/n_total*100:.2f} %)")
print(f"Pacientes con k<5: {n_vulnerable_k5} ({n_vulnerable_k5/n_total*100:.2f} %)")

## 4. Reducción de dimensionalidad

La cardinalidad original de los códigos CIE-9 produce una matriz One-Hot
Encoding de 2.405 dimensiones, incompatible con la inyección de ruido en
Privacidad Diferencial. Se agrupan los códigos en nueve macrocategorías
médicas mediante una función de mapeo basada en el conocimiento del
dominio, reduciendo el espacio a 121 dimensiones.

In [ ]:
def map_diagnosis(code):
    if pd.isna(code) or code == "?":
        return "Other"
    if isinstance(code, str) and (code.startswith("V") or code.startswith("E")):
        return "Other"
    try:
        v = float(code)
    except (TypeError, ValueError):
        return "Other"
    if 250 <= v < 251:                  return "Diabetes"
    if 390 <= v <= 459 or v == 785:     return "Circulatory"
    if 460 <= v <= 519 or v == 786:     return "Respiratory"
    if 520 <= v <= 579 or v == 787:     return "Digestive"
    if 580 <= v <= 629 or v == 788:     return "Genitourinary"
    if 710 <= v <= 739:                 return "Musculoskeletal"
    if 800 <= v <= 999:                 return "Injury"
    if 140 <= v <= 239:                 return "Neoplasms"
    return "Other"

for column in ("diag_1", "diag_2", "diag_3"):
    df[f"{column}_category"] = df[column].apply(map_diagnosis)

df_reduced = df.drop(columns=["diag_1", "diag_2", "diag_3", "medical_specialty"])
X_raw = df_reduced.drop(columns=["readmitted"])
y_bin = (df_reduced["readmitted"] != "NO").astype(int)
X_encoded = pd.get_dummies(X_raw, drop_first=True)
print(f"Dimensiones tras reducción semántica: {X_encoded.shape}")

## 5. Partición estratificada y modelos baseline

El conjunto se divide en 80 % entrenamiento y 20 % prueba con muestreo
estratificado. Se entrenan cuatro modelos clásicos de aprendizaje
automático para establecer la cota máxima de utilidad sin privacidad.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y_bin, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y_bin
)
scaler = StandardScaler().fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)


def build_baseline_models():
    return {
        "Regresión Logística":    LogisticRegression(max_iter=1000, random_state=RANDOM_STATE, class_weight="balanced"),
        "Random Forest":          RandomForestClassifier(n_estimators=100, max_depth=10, random_state=RANDOM_STATE, class_weight="balanced"),
        "Árbol de Decisión":      DecisionTreeClassifier(max_depth=10, random_state=RANDOM_STATE, class_weight="balanced"),
        "Naive Bayes (Gaussian)": GaussianNB(),
    }


baseline_rows = []
for name, model in build_baseline_models().items():
    model.fit(X_train_scaled, y_train)
    predictions = model.predict(X_test_scaled)
    baseline_rows.append({
        "Modelo": name,
        "Accuracy": accuracy_score(y_test, predictions),
        "F1-Score": f1_score(y_test, predictions, average="weighted"),
        "k": 0,
    })

df_baseline = pd.DataFrame(baseline_rows).sort_values("F1-Score", ascending=False)
display(df_baseline.round(4))

## 6. Exportación para ARX Desktop

Para la fase de $k$-anonimidad se exportan los conjuntos de entrenamiento
y prueba al formato esperado por ARX (separador `;`) junto con las
cinco jerarquías de generalización descritas en la memoria.

In [ ]:
train_index, test_index = X_train.index, X_test.index
df_train_raw = df_reduced.loc[train_index].copy()
df_test_raw = df_reduced.loc[test_index].copy()

for column in ("admission_type_id", "time_in_hospital"):
    df_train_raw[column] = df_train_raw[column].astype(str)
    df_test_raw[column] = df_test_raw[column].astype(str)

df_train_raw.to_csv(WORKDIR / "arx_train.csv", sep=";", index=False)
df_test_raw.to_csv(WORKDIR / "arx_test.csv", sep=";", index=False)

In [ ]:
os.makedirs(WORKDIR / "arx_hierarchies", exist_ok=True)

hierarchies = {
    "race.csv": """Caucasian;Caucasian;White;*
AfricanAmerican;AfricanAmerican;NonWhite;*
Hispanic;Hispanic;NonWhite;*
Asian;Asian;NonWhite;*
Other;Other;NonWhite;*""",
    "gender.csv": """Female;*
Male;*
Unknown/Invalid;*""",
    "age.csv": "\n".join([
        "[0-10);[0-20);[0-40);*", "[10-20);[0-20);[0-40);*",
        "[20-30);[20-40);[0-40);*", "[30-40);[20-40);[0-40);*",
        "[40-50);[40-60);[40-80);*", "[50-60);[40-60);[40-80);*",
        "[60-70);[60-80);[40-80);*", "[70-80);[60-80);[40-80);*",
        "[80-90);[80-100);[80-100);*", "[90-100);[80-100);[80-100);*",
    ]),
    "admission_type_id.csv": """1;Urgente;Conocido;*
2;Urgente;Conocido;*
3;Electivo;Conocido;*
4;Newborn;Conocido;*
5;Desconocido;Desconocido;*
6;Desconocido;Desconocido;*
7;Urgente;Conocido;*
8;Desconocido;Desconocido;*""",
}

time_lines = []
for i in range(1, 15):
    if i in (1, 2):       l1, l2 = "[1-2]", "[1-7]"
    elif i in (3, 4):     l1, l2 = "[3-4]", "[1-7]"
    elif i in (5, 6, 7):  l1, l2 = "[5-7]", "[1-7]"
    elif i in (8, 9, 10): l1, l2 = "[8-10]", "[8-14]"
    else:                 l1, l2 = "[11-14]", "[8-14]"
    time_lines.append(f"{i};{l1};{l2};*")
hierarchies["time_in_hospital.csv"] = "\n".join(time_lines)

for filename, content in hierarchies.items():
    (WORKDIR / "arx_hierarchies" / filename).write_text(content)

La anonimización propiamente dicha se ejecuta de forma manual en ARX
Desktop. El procedimiento detallado figura en `docs/guia_arx.md`. Una vez
exportados los cinco archivos `arx_output_k<k>.csv`, se reanuda la
ejecución del cuaderno.

## 7. Evaluación post-ARX

In [ ]:
def evaluate_kanon(filepath, k_value):
    df_anon = pd.read_csv(filepath, sep=";")
    n_initial = len(df_anon)
    mask_suppressed = df_anon["race"].astype(str).str.fullmatch(r"\*")
    df_anon = df_anon[~mask_suppressed].copy()
    suppression_pct = (n_initial - len(df_anon)) / n_initial * 100

    y_anon = (df_anon["readmitted"] != "NO").astype(int)
    X_anon = df_anon.drop(columns=["readmitted"])
    X_test_raw_view = df_test_raw.drop(columns=["readmitted"])

    n_anon = len(X_anon)
    stacked = pd.concat([X_anon, X_test_raw_view], ignore_index=True)
    stacked_encoded = pd.get_dummies(stacked, drop_first=True)
    X_anon_array = stacked_encoded.iloc[:n_anon]
    X_test_array = stacked_encoded.iloc[n_anon:]

    sc = StandardScaler().fit(X_anon_array)
    X_anon_scaled = sc.transform(X_anon_array)
    X_test_scaled_local = sc.transform(X_test_array)

    y_test_local = (df_test_raw["readmitted"] != "NO").astype(int).values

    rows = []
    for name, model in build_baseline_models().items():
        model.fit(X_anon_scaled, y_anon)
        predictions = model.predict(X_test_scaled_local)
        rows.append({
            "Modelo": name,
            "Accuracy": accuracy_score(y_test_local, predictions),
            "F1-Score": f1_score(y_test_local, predictions, average="weighted"),
            "k": k_value,
            "pct_suprimido": round(suppression_pct, 2),
            "filas_efectivas": len(df_anon),
            "columnas_OHE": X_anon_array.shape[1],
        })
    return pd.DataFrame(rows)


results_kanon = [df_baseline.assign(
    pct_suprimido=0, filas_efectivas=len(X_train), columnas_OHE=X_encoded.shape[1]
)]
for k in K_VALUES:
    filepath = WORKDIR / f"arx_output_k{k}.csv"
    if filepath.exists():
        results_kanon.append(evaluate_kanon(filepath, k))

df_kanon = pd.concat(results_kanon, ignore_index=True)
df_kanon.to_csv(WORKDIR / "resultados_kanon.csv", index=False)
display(df_kanon.pivot(index="Modelo", columns="k", values="Accuracy").round(4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for model_name in df_kanon["Modelo"].unique():
    subset = df_kanon[df_kanon["Modelo"] == model_name].sort_values("k")
    axes[0].plot(subset["k"], subset["Accuracy"], marker="o", label=model_name)
    axes[1].plot(subset["k"], subset["F1-Score"], marker="s", label=model_name)

for ax, ylabel, title in [
    (axes[0], "Accuracy", "Degradación de Accuracy con k"),
    (axes[1], "F1-Score", "Degradación de F1-Score con k"),
]:
    ax.set_xlabel("k"); ax.set_ylabel(ylabel); ax.set_title(title)
    ax.set_xscale("symlog")
    ax.set_xticks([0] + K_VALUES); ax.set_xticklabels(["0"] + [str(k) for k in K_VALUES])
    ax.grid(alpha=0.3); ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(WORKDIR / "comparativa_kanon.png", dpi=180, bbox_inches="tight")
plt.show()

## 8. Aplicación de Privacidad Diferencial

Se ejecuta el sweep $\\epsilon \\in \\{0,1\\ 0,5\\ 1\\ 5\\ 10\\}$ con cinco repeticiones por
punto sobre la Regresión Logística y el Naive Bayes Gaussian. La cota
de sensibilidad $\\texttt{data\\_norm}$ se fija al percentil 95 de las normas $L_2$
del conjunto de entrenamiento escalado para evitar la sobrecalibración
del ruido por filas atípicas.

In [ ]:
data_norm = float(np.percentile(np.linalg.norm(X_train_scaled, axis=1), 95))
mins = np.percentile(X_train_scaled, 1, axis=0)
maxs = np.percentile(X_train_scaled, 99, axis=0)
print(f"data_norm (p95): {data_norm:.4f}")

dp_rows = []
for epsilon in EPSILON_VALUES:
    for rep in range(N_REPETITIONS_DP):
        seed = RANDOM_STATE + rep
        lr_dp = dpm.LogisticRegression(epsilon=epsilon, data_norm=data_norm, max_iter=100, random_state=seed)
        lr_dp.fit(X_train_scaled, y_train)
        predictions = lr_dp.predict(X_test_scaled)
        dp_rows.append({
            "modelo": "Regresión Logística", "epsilon": epsilon, "rep": rep,
            "Accuracy": accuracy_score(y_test, predictions),
            "F1-Score": f1_score(y_test, predictions, average="weighted"),
        })

        nb_dp = dpm.GaussianNB(epsilon=epsilon, bounds=(mins, maxs), random_state=seed)
        nb_dp.fit(X_train_scaled, y_train)
        predictions = nb_dp.predict(X_test_scaled)
        dp_rows.append({
            "modelo": "Naive Bayes (Gaussian)", "epsilon": epsilon, "rep": rep,
            "Accuracy": accuracy_score(y_test, predictions),
            "F1-Score": f1_score(y_test, predictions, average="weighted"),
        })

df_dp = pd.DataFrame(dp_rows)
df_dp.to_csv(WORKDIR / "resultados_dp.csv", index=False)

dp_aggregated = df_dp.groupby(["modelo", "epsilon"]).agg(
    Acc_mean=("Accuracy", "mean"), Acc_std=("Accuracy", "std"),
    F1_mean=("F1-Score", "mean"), F1_std=("F1-Score", "std"),
).round(4)
display(dp_aggregated)

## 9. Auditoría adversarial: MIA Black-Box

In [ ]:
def run_mia(model, X_train_array, y_train_array, X_test_array, y_test_array, label):
    art_classifier = ScikitlearnLogisticRegression(model=model)
    attack = MembershipInferenceBlackBox(estimator=art_classifier, attack_model_type="rf")
    rng = np.random.default_rng(RANDOM_STATE)

    n_tr = min(len(X_train_array), 10000); n_te = min(len(X_test_array), 5000)
    idx_tr = rng.choice(len(X_train_array), n_tr, replace=False)
    idx_te = rng.choice(len(X_test_array), n_te, replace=False)

    Xa = X_train_array[idx_tr]; ya = np.asarray(y_train_array)[idx_tr]
    Xt = X_test_array[idx_te];  yt = np.asarray(y_test_array)[idx_te]
    half_tr = n_tr // 2; half_te = n_te // 2

    attack.fit(x=Xa[:half_tr], y=ya[:half_tr], test_x=Xt[:half_te], test_y=yt[:half_te])
    inferred_members = attack.infer(Xa[half_tr:], ya[half_tr:])
    inferred_non = attack.infer(Xt[half_te:], yt[half_te:])

    attack_acc = ((inferred_members == 1).sum() + (inferred_non == 0).sum()) / (len(inferred_members) + len(inferred_non))
    advantage = float((inferred_members == 1).mean() - (inferred_non == 1).mean())
    utility = float(accuracy_score(y_test_array, model.predict(X_test_array)))
    return {"escenario": label, "utility_acc": round(utility, 4),
            "attack_acc": round(float(attack_acc), 4), "attack_advantage": round(advantage, 4)}


mia_rows = []
lr_baseline = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE, class_weight="balanced").fit(X_train_scaled, y_train)
mia_rows.append(run_mia(lr_baseline, X_train_scaled, y_train.values, X_test_scaled, y_test.values, "Baseline LR"))

for epsilon in (0.1, 10.0):
    lr_dp = dpm.LogisticRegression(epsilon=epsilon, data_norm=data_norm, max_iter=100, random_state=RANDOM_STATE)
    lr_dp.fit(X_train_scaled, y_train)
    mia_rows.append(run_mia(lr_dp, X_train_scaled, y_train.values, X_test_scaled, y_test.values, f"LR DP epsilon={epsilon}"))

df_mia = pd.DataFrame(mia_rows)
df_mia.to_csv(WORKDIR / "mia_results.csv", index=False)
display(df_mia)

## 10. Tests de significancia estadística

Los tests de McNemar contrastan las predicciones del baseline frente a
cada modelo $k$-anonimizado. Los tests de Wilcoxon de una muestra
contrastan la Accuracy promedio de los modelos diferencialmente privados
frente al baseline sin privacidad.

In [ ]:
def mcnemar_test(y_true, predictions_a, predictions_b):
    correct_a = (predictions_a == y_true); correct_b = (predictions_b == y_true)
    b = int(((correct_a) & (~correct_b)).sum()); c = int(((~correct_a) & (correct_b)).sum())
    if b + c == 0:
        return 1.0, 0, 0, 0.0
    chi_squared = (abs(b - c) - 1) ** 2 / (b + c)
    return float(1 - chi2.cdf(chi_squared, df=1)), b, c, float(chi_squared)


baseline_predictions = {}
for name, model in build_baseline_models().items():
    model.fit(X_train_scaled, y_train)
    baseline_predictions[name] = model.predict(X_test_scaled)

mcnemar_rows = []
for k in K_VALUES:
    filepath = WORKDIR / f"arx_output_k{k}.csv"
    if not filepath.exists():
        continue
    df_k = pd.read_csv(filepath, sep=";")
    df_k = df_k[~df_k["race"].astype(str).str.fullmatch(r"\*")]
    y_k = (df_k["readmitted"] != "NO").astype(int).values
    X_k = df_k.drop(columns=["readmitted"])
    X_test_view = df_test_raw.drop(columns=["readmitted"])
    n_k = len(X_k)
    encoded = pd.get_dummies(pd.concat([X_k, X_test_view], ignore_index=True), drop_first=True)
    Xa_array = encoded.iloc[:n_k]; Xt_array = encoded.iloc[n_k:]
    sc_local = StandardScaler().fit(Xa_array)
    for name, model in build_baseline_models().items():
        model.fit(sc_local.transform(Xa_array), y_k)
        predictions_k = model.predict(sc_local.transform(Xt_array))
        p_value, b, c, chi_sq = mcnemar_test(y_test.values, baseline_predictions[name], predictions_k)
        mcnemar_rows.append({
            "modelo": name, "k": k, "b": b, "c": c,
            "chi2": chi_sq, "p_value": p_value, "significativo_005": p_value < 0.05,
        })

df_mcnemar = pd.DataFrame(mcnemar_rows)
df_mcnemar.to_csv(WORKDIR / "mcnemar_kanon.csv", index=False)
display(df_mcnemar.round(4))

In [ ]:
baselines_global = {"Regresión Logística": 0.6178, "Naive Bayes (Gaussian)": 0.5912}
wilcoxon_rows = []
for model_name, baseline_acc in baselines_global.items():
    for epsilon in EPSILON_VALUES:
        sample = df_dp[(df_dp.modelo == model_name) & (df_dp.epsilon == epsilon)]["Accuracy"].values
        try:
            statistic, p_value = wilcoxon(sample - baseline_acc)
        except ValueError:
            p_value = 1.0
        wilcoxon_rows.append({
            "modelo": model_name, "epsilon": epsilon,
            "mean_acc": float(np.mean(sample)), "baseline": baseline_acc,
            "p_value": float(p_value),
            "delta_pct": (float(np.mean(sample)) - baseline_acc) / baseline_acc * 100,
        })

df_wilcoxon = pd.DataFrame(wilcoxon_rows)
df_wilcoxon.to_csv(WORKDIR / "wilcoxon_dp.csv", index=False)
display(df_wilcoxon.round(4))

## 11. Artefactos generados

El cuaderno produce las tablas y figuras consumidas por la memoria:

| Archivo | Descripción |
|---|---|
| `resultados_kanon.csv` | Métricas por modelo y nivel de $k$ |
| `resultados_dp.csv`    | 50 ejecuciones del sweep ε × 5 repeticiones |
| `mcnemar_kanon.csv`    | Tests de significancia ARX |
| `wilcoxon_dp.csv`      | Tests de significancia DP |
| `mia_results.csv`      | Auditoría adversarial Black-Box |
| `comparativa_kanon.png` | Curvas de degradación ARX |
